# LangGraph
从线性的 Chain（链）跨入支持循环、分支与复杂状态管理的 Graph（图）智能体架构！
在传统的 Prompt 链或单向流水线（Chain）中，程序的执行是线性的（Linear）。但在构建复杂的真实企业级 Agent 时，我们需要循环（Loops）、条件分支（Conditional Edges）、状态 Persistence（持久化记忆） 以及 人工干预（Human-in-the-loop）

## LangGraph 状态图与有向有环图架构
1. **突破线性 Chain 限制**：理解为什么传统 LLM Chain 无法应对复杂的 Agent 迭代，掌握有向有环图（DAG / Graph with Cycles）的设计思想。
2. **攻克 LangGraph 四大核心要素**：
    * State（全局状态）：作为状态机在各个节点间传递的数据载体。
    * Nodes（节点）：处理 State 的 Python 函数（调用 LLM、执行 Tool 或更新数据）。
    * Edges（普通边）：定义节点之间的固定流转路线。
    * Conditional Edges（条件边）：根据 LLM 的输出决定下一步走哪个分支（如：“继续调用工具”还是“回答用户”）
3. 利用 LangGraph 从零搭建一个纯粹的循环 ReAct Agent。


## LangGraph 的底层逻辑与图架构
1. 为什么从 Chain 演进到 Graph？
    * 线性链（Chain）：`Input -> Prompt -> LLM -> OutputParser -> Output`（无法回头，无法根据中间结果调整策略）。
    * 图架构（Graph）：
        * 允许循环（Cycles）：Agent 执行 Tool 失败后，可以带着 Error 信息重新送回 LLM 重新思考（Re-try / Self-Correction）。
        * 状态透明（State Machine）：全局维持一个强类型的 `TypedDict` 或 `Pydantic` 对象，每一个 Node 都可以读取并增量更新（Annotated Reducer）该状态。

#### LangGraph 的四大构件
* State（全局状态定义）：

In [ ]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    # operator.add 表示新产生的 messages 会追加（Append）到列表中，而不是覆写（Overwrite）
    messages: Annotated[Sequence[BaseMessage], operator.add]

* Nodes（节点）：
节点就是普通的 Python 函数，接受当前的 `State`，处理完后返回更新后的字典。

In [ ]:
def call_model(state: AgentState):
    messages = state['messages']
    response = model.invoke(messages)
    return {"messages": [response]}

*  普通边Edges：从 action 节点执行完后，必须循环回到 agent 节点

    `workflow.add_edge("action", "agent")`

* Conditional Edges（条件边）：动态路由函数，根据当前 State 判断下一个到达的 Node。

In [ ]:
def should_continue(state: AgentState):
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "action" # 路由到工具节点
    return "end"        # 路由到结束节点


下面的代码演示如何使用 LangGraph 组装一个包含状态管理、工具调用与条件路由循环的完整图智能体：

In [ ]:
from typing import Annotated, TypedDict, Sequence
import operator

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# --- 1. 定义工具 (Tool) ---
@tool
def calculate_salary_after_tax(base_salary: float) -> str:
    """计算扣税后的预估税后薪资（仅用于演示）"""
    tax = base_salary * 0.2
    return f"基础薪资 {base_salary} 元，预估扣税 {tax} 元，税后到手大约 {base_salary - tax} 元。"

tools = [calculate_salary_after_tax]
# 创建工具字典便于节点内查表调用
tool_map = {t.name: t for t in tools}

# --- 2. 定义 AgentState (全局状态) ---
class AgentState(TypedDict):
    # 使用 operator.add 增量追加消息历史
    messages: Annotated[Sequence[BaseMessage], operator.add]

# --- 3. 定义图节点 (Nodes) ---

# A. LLM 思考节点
def agent_node(state: AgentState):
    # 模拟绑定了工具的 LLM (在真实环境下，可替换为 llm.bind_tools(tools))
    messages = state["messages"]
    last_message = messages[-1]

    # 逻辑模拟：如果是首次提问，产生 Tool Call；如果已经拿到 Tool 回复，生成最终文本
    if len(messages) == 1:
        # 模拟模型决定调用 calculate_salary_after_tax 工具
        from langchain_core.messages import AIMessage
        ai_msg = AIMessage(
            content="",
            tool_calls=[{
                "name": "calculate_salary_after_tax",
                "args": {"base_salary": 20000.0},
                "id": "call_001"
            }]
        )
        return {"messages": [ai_msg]}
    else:
        # 模拟模型根据工具结果生成最终回复
        from langchain_core.messages import AIMessage
        ai_msg = AIMessage(content="根据计算，您的税后薪资大约为 16000.0 元。")
        return {"messages": [ai_msg]}

# B. 工具执行节点 (Tool Execution Node)
def action_node(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]

    tool_outputs = []
    for tool_call in last_message.tool_calls:
        tool_obj = tool_map[tool_call["name"]]
        # 执行工具
        output = tool_obj.invoke(tool_call["args"])
        # 构造 ToolMessage 挂回消息链
        tool_outputs.append(
            ToolMessage(content=str(output), tool_call_id=tool_call["id"])
        )
    return {"messages": tool_outputs}

# --- 4. 定义条件边分支路由 (Conditional Edge Router) ---
def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    last_message = messages[-1]
    # 如果模型输出了 tool_calls，则跳转到 action 节点，否则结束图流程
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "action"
    return "end"

# --- 5. 构建与编译 StateGraph ---
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("agent", agent_node)
workflow.add_node("action", action_node)

# 设置入口节点
workflow.set_entry_point("agent")

# 添加条件边：从 agent 节点出发，根据 should_continue 判断去向
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "action": "action",
        "end": END
    }
)

# 添加普通边：从 action 节点执行完后，必须循环回到 agent 节点
workflow.add_edge("action", "agent")

# 编译生成可执行图对象
app = workflow.compile()

# --- 6. 执行图工作流 ---
if __name__ == "__main__":
    print("🚀 正在运行 LangGraph 循环状态图...\n")

    initial_input = {"messages": [HumanMessage(content="我的税前薪资是 20000，帮我算算税后到手多少？")]}

    # 执行图并观察每个节点的状态演变
    for chunk in app.stream(initial_input):
        for node_name, state_update in chunk.items():
            print(f"📍 [节点完成]: {node_name}")
            print(f"   ↳ 最新增量消息: {state_update['messages'][-1].content or state_update['messages'][-1].tool_calls}\n")

1. State Reducer 机制（`operator.add` vs. 覆写）：
    * 在定义 `AgentState` 时，我们给 `messages` 字段标记了 `Annotated[..., operator.add]`。
    * 工程思考：如果不写 `operator.add`，当节点 `action_node` 返回 `{"messages": [ToolMessage(...)]}` 时，全局状态里的 `messages` 列表会发生什么变化？这种设计对于支持多轮对话上下文有什么重要作用？

2. 死循环防御（Max Iterations）：
    * 如果 LLM 在某个工具节点调用失败后反复重试，可能会导致图在 `agent` 和 `action` 节点之间无限循环，消耗大量 Token。
    * 思考：在 `LangGraph` 中，除了在图编译配置中限制 `recursion_limit`（递归上限）之外，你可以在 `AgentState` 中增加什么自定义状态字段（如 `retry_count`）来主动阻断死循环？
